# SBA loan data cleaning

This is the data cleaning part of our SBA loan case study. I used the original `SBAnational.csv` and checked for missing values, inconsistent formats, and a few fields that looked off. The goal is to give everyone on the team the same cleaned data instead of each of us fixing things differently.

We're looking at two loan outcomes: **paid in full (`PIF`)** and **charged off (`CHGOFF`)**. For the model, `Default = 1` means charged off and `Default = 0` means paid in full. Loans with no recorded outcome stay unlabeled.

The model is supposed to estimate risk **at loan approval**, so I also made sure its input file doesn't include things we would only learn after a loan was approved.

## Files this notebook makes

There are two main outputs:

- **`data/processed/sba_eda_ready.csv`** is for charts and financial analysis. It keeps all the loans and includes the historical loan information.
- **`data/processed/sba_model_candidates.csv`** is for logistic regression. It only keeps loans with a known outcome and the columns we're considering as inputs.

I also saved the missing-value counts, data checks, and a list of model columns in **`outputs/cleaning_audit/`**. Those smaller files are useful if anyone wants to see why something was changed or left alone.

**To run this:** Open the repo in VS Code or Jupyter, put `SBAnational.csv` in `data/raw/`, and run the cells in order. If you keep the CSV somewhere else, set `RAW_CSV_OVERRIDE` below. The raw file isn't changed.

## Quick look at the data

These are the counts from our original SBA file. The code below checks them again when it runs.

| Check | Loans or records |
| --- | ---: |
| Total loans | 899,164 |
| Paid in full | 739,609 |
| Charged off | 157,558 |
| Missing loan outcome | 1,997 |
| Loans available for modeling | 897,167 |
| Duplicate loan IDs | 0 |
| Unknown or zero industry codes | 201,948 |
| Loan terms of zero or less | 810 |

I didn't delete a row just because a value was missing or unusual. The later audit section shows what needs another look.

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

# Only change this if your CSV is saved somewhere else.
RAW_CSV_OVERRIDE = None  # Example: r"C:\Users\me\Downloads\SBAnational.csv"

cwd = Path.cwd().resolve()
search_roots = [cwd, *list(cwd.parents)[:3]]
repo_root = next(
    (p for p in search_roots if (p / 'notebooks').is_dir() and (p / 'data').is_dir()),
    cwd
)

paths_to_try = (
    [Path(RAW_CSV_OVERRIDE).expanduser()] if RAW_CSV_OVERRIDE
    else [repo_root / 'data' / 'raw' / 'SBAnational.csv',
          repo_root / 'SBAnational.csv', cwd / 'SBAnational.csv']
)
raw_path = next((p for p in paths_to_try if p.is_file()), None)
if raw_path is None:
    raise FileNotFoundError(
        'SBAnational.csv not found. Place it in data/raw/ or set RAW_CSV_OVERRIDE.\n'
        + '\n'.join(str(p) for p in paths_to_try)
    )

processed_dir = repo_root / 'data' / 'processed'
audit_dir = repo_root / 'outputs' / 'cleaning_audit'
processed_dir.mkdir(parents=True, exist_ok=True)
audit_dir.mkdir(parents=True, exist_ok=True)
print('Input:', raw_path)
print('Project root:', repo_root)

## Read the original CSV

First, the code loads the file and checks that the expected columns are there. Loan IDs and ZIP codes are read as text so numbers like `00501` don't lose their leading zeros.

In [ ]:
CODE_COLUMNS = {
    'LoanNr_ChkDgt': 'string',
    'Zip': 'string',
    'NAICS': 'string',
    'ApprovalFY': 'string',
    'NewExist': 'string',
    'UrbanRural': 'string',
    'FranchiseCode': 'string',
    'LowDoc': 'string',
    'RevLineCr': 'string',
}
try:
    raw = pd.read_csv(raw_path, dtype=CODE_COLUMNS, encoding='utf-8', low_memory=False)
except UnicodeDecodeError:
    raw = pd.read_csv(raw_path, dtype=CODE_COLUMNS, encoding='latin-1', low_memory=False)

expected = [
    'LoanNr_ChkDgt','Name','City','State','Zip','Bank','BankState',
    'NAICS','ApprovalDate','ApprovalFY','Term','NoEmp','NewExist',
    'CreateJob','RetainedJob','FranchiseCode','UrbanRural','RevLineCr',
    'LowDoc','ChgOffDate','DisbursementDate','DisbursementGross',
    'BalanceGross','MIS_Status','ChgOffPrinGr','GrAppv','SBA_Appv'
]
missing_cols = sorted(set(expected) - set(raw.columns))
if missing_cols:
    raise ValueError('Unexpected source schema: missing ' + str(missing_cols))
if len(raw.columns) != 27:
    print('WARNING: Expected 27 columns, found', len(raw.columns))

n_original = len(raw)
print(f'Loaded {n_original:,} records, {raw.shape[1]} columns.')
print('Duplicate loan IDs:', int(raw['LoanNr_ChkDgt'].duplicated(keep=False).sum()))
display(raw.head(3)[['LoanNr_ChkDgt','ApprovalDate','ApprovalFY','Term','MIS_Status','GrAppv']])

## Check the missing values

Before changing anything, I counted the blanks in each column and checked the values used for fields like `LowDoc` and `RevLineCr`. Some of those codes aren't a simple yes or no, so I wanted to see what was actually in the file first.

The counts are saved in the audit folder for anyone who wants to check them.

In [ ]:
# Count blank cells, including cells with only spaces.
def blank_count(s):
    if pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s):
        return int(s.isna().sum() + s.fillna('').astype(str).str.strip().eq('').sum() - s.isna().sum())
    return int(s.isna().sum())

raw_audit = pd.DataFrame({
    'column': raw.columns,
    'dtype_original': [str(raw[c].dtype) for c in raw.columns],
    'missing_n': [blank_count(raw[c]) for c in raw.columns],
    'unique_nonnull': [raw[c].nunique(dropna=True) for c in raw.columns]
})
raw_audit['missing_pct'] = (100 * raw_audit.missing_n / n_original).round(3)
raw_audit.sort_values('missing_pct', ascending=False).to_csv(
    audit_dir / '01_original_missingness.csv', index=False
)

for c in ['MIS_Status', 'LowDoc', 'RevLineCr', 'NewExist', 'UrbanRural']:
    profile = raw[c].fillna('<MISSING>').astype(str).str.strip().value_counts(dropna=False)
    profile.rename_axis('original_value').reset_index(name='loans').to_csv(
        audit_dir / ('02_codes_' + c + '.csv'), index=False
    )
print('Most-missing source fields:')
display(raw_audit.sort_values('missing_pct', ascending=False).head(10))
print('Original statuses:')
display(raw['MIS_Status'].fillna('<MISSING>').value_counts(dropna=False).to_frame('loans'))

## Clean the columns

This is where most of the actual cleaning happens. Dollar amounts become numbers, dates are put into a consistent format, and a few coded fields get clearer labels.

A couple things to know:

- `NAICS = 0` doesn't identify an industry, so it becomes **Unknown**.
- For `LowDoc`, `RevLineCr`, and other categories, I only assign a label when the code is clear. The rest stay **Unknown**.
- `ApprovalFY` is a **federal fiscal year**, so an October approval can be listed under the following year. I also fixed a handful of older dates that Python read as being in the 2060s.
- I kept the actual disbursement and charge-off information for looking back at loans, but not for predicting risk at approval.

At this point, we're fixing formats and adding useful columns, not removing loans.

In [ ]:
# Borrower and bank names aren't needed in the cleaned files.
eda = raw.drop(columns=['Name', 'Bank']).copy()

# Keep the loan ID for matching rows, not for predicting defaults.
eda['LoanNr_ChkDgt'] = eda['LoanNr_ChkDgt'].astype('string').str.strip()

# Clean up the loan statuses. A blank status is still unknown.
status_raw = eda['MIS_Status'].astype('string').str.upper().str.replace(r'\s+', '', regex=True)
status_normalized = status_raw.map({'PIF': 'PIF', 'CHGOFF': 'CHGOFF'})
eda['MIS_Status'] = status_normalized.astype('string')
eda['Default'] = pd.Series(pd.NA, index=eda.index, dtype='Int8')
eda.loc[eda['MIS_Status'].eq('PIF').fillna(False), 'Default'] = 0
eda.loc[eda['MIS_Status'].eq('CHGOFF').fillna(False), 'Default'] = 1

# Convert dollar amounts to numbers. Later loan activity stays out of the model.
money_fields = ['DisbursementGross', 'BalanceGross', 'ChgOffPrinGr', 'GrAppv', 'SBA_Appv']
for col in money_fields:
    s = (eda[col].astype('string')
         .str.replace(r'[$,\s]', '', regex=True)
         .replace('', pd.NA))
    eda[col] = pd.to_numeric(s, errors='coerce').astype('float64')

# The original dates use a two-digit year.
date_cols = ['ApprovalDate', 'DisbursementDate', 'ChgOffDate']
date_century_corrections = {}
for col in date_cols:
    parsed = pd.to_datetime(eda[col], format='%d-%b-%y', errors='coerce')
    # Some older dates get read as 2060s dates. These belong in the 1960s.
    wrong_century = parsed.dt.year.gt(2026)
    date_century_corrections[col] = int(wrong_century.sum())
    if wrong_century.any():
        parsed.loc[wrong_century] = parsed.loc[wrong_century] - pd.DateOffset(years=100)
    eda[col] = parsed

# Pull out the fiscal year, but still track entries with unusual formatting.
fy_raw = raw['ApprovalFY'].astype('string').str.strip()
eda['ApprovalFY'] = pd.to_numeric(
    fy_raw.str.extract(r'(\d{4})', expand=False), errors='coerce'
).astype('Int64')
eda['ApprovalYear'] = eda['ApprovalDate'].dt.year.astype('Int64')
eda['ApprovalYear'] = eda['ApprovalYear'].fillna(eda['ApprovalFY'])
eda['ApprovalMonth'] = eda['ApprovalDate'].dt.month.astype('Int64')

# Keep ZIP codes at five digits.
zip_raw = eda['Zip'].astype('string').str.strip()
eda['ZIP5'] = zip_raw.where(zip_raw.str.fullmatch(r'\d{1,5}').fillna(False))
eda['ZIP5'] = eda['ZIP5'].str.zfill(5)

# Leave missing or unusual state codes as Unknown.
for col in ['State', 'BankState']:
    s = eda[col].astype('string').str.upper().str.strip()
    eda[col] = s.where(s.str.fullmatch(r'[A-Z]{2}').fillna(False), 'Unknown')
known_states = eda['State'].ne('Unknown') & eda['BankState'].ne('Unknown')
eda['InState_Bank'] = pd.Series(pd.NA, index=eda.index, dtype='Int8')
eda.loc[known_states, 'InState_Bank'] = (
    eda.loc[known_states, 'State'].eq(eda.loc[known_states, 'BankState'])
).astype('int8')

# NAICS code 0 does not give us an industry.
naics_raw = eda['NAICS'].astype('string').str.strip()
naics_valid = naics_raw.str.fullmatch(r'\d{2,6}').fillna(False) & ~naics_raw.str.fullmatch(r'0+').fillna(False)
eda['NAICS_Sector'] = naics_raw.str[:2].where(naics_valid, 'Unknown')

newex = eda['NewExist'].astype('string').str.strip()
eda['NewExist_clean'] = newex.map({'1': 'Existing', '2': 'New'}).fillna('Unknown')
urbr = eda['UrbanRural'].astype('string').str.strip()
eda['UrbanRural_clean'] = urbr.map({'1': 'Urban', '2': 'Rural'}).fillna('Unknown')

for source, target in [('LowDoc', 'LowDoc_clean'), ('RevLineCr', 'RevLineCr_clean')]:
    z = eda[source].astype('string').str.upper().str.strip()
    eda[target] = z.where(z.isin(['Y', 'N']).fillna(False), 'Unknown')

# Keep unusual counts and flag them later rather than deleting rows.
for col in ['Term', 'NoEmp', 'CreateJob', 'RetainedJob']:
    eda[col] = pd.to_numeric(eda[col], errors='coerce')

# Use this only if the guarantee amount is known at the time of prediction.
eda['SBA_Guarantee_Share'] = (
    eda['SBA_Appv'] / eda['GrAppv'].where(eda['GrAppv'].gt(0))
)
eda.loc[~eda['SBA_Guarantee_Share'].between(0, 1), 'SBA_Guarantee_Share'] = np.nan

print('Cleaned example (no row exclusions yet):')
display(eda[['LoanNr_ChkDgt','MIS_Status','Default','ApprovalYear',
             'NAICS_Sector','GrAppv','SBA_Appv','SBA_Guarantee_Share']].head(5))

## Flag anything that needs a second look

Some entries still look unusual, like a loan term of zero or an unknown industry code. The code counts these and puts them in `03_quality_checks.csv` instead of dropping those loans right away.

That gives us something concrete to check later if a chart or model result looks strange.

In [ ]:
# Count suspicious values without automatically removing the loans.
raw_status_nonempty = status_raw.notna() & status_raw.ne('')
unknown_status = raw_status_nonempty & eda['MIS_Status'].isna()
fy_malformed = fy_raw.notna() & fy_raw.ne('') & ~fy_raw.str.fullmatch(r'\d{4}').fillna(False)
# ApprovalFY uses the federal fiscal year.
# October to December approvals usually count toward the next fiscal year.
expected_fiscal_year = (
    eda['ApprovalDate'].dt.year.astype('Int64')
    + eda['ApprovalDate'].dt.month.ge(10).astype('Int64')
)
year_mismatch = (
    eda['ApprovalDate'].notna() & eda['ApprovalFY'].notna()
    & expected_fiscal_year.ne(eda['ApprovalFY'])
)

audit_checks = {
    'raw_records': n_original,
    'unique_loan_ids': int(eda['LoanNr_ChkDgt'].nunique(dropna=True)),
    'duplicate_loan_id_rows': int(eda['LoanNr_ChkDgt'].duplicated(keep=False).sum()),
    'paid_in_full_PIF': int(eda['Default'].eq(0).sum()),
    'charged_off_CHGOFF': int(eda['Default'].eq(1).sum()),
    'excluded_from_supervised_model_missing_or_unknown_target': int(eda['Default'].isna().sum()),
    'unknown_nonempty_status_codes': int(unknown_status.sum()),
    'NAICS_zero_or_unknown': int(eda['NAICS_Sector'].eq('Unknown').sum()),
    'Term_zero_or_negative': int(eda['Term'].le(0).sum()),
    'NoEmp_negative': int(eda['NoEmp'].lt(0).sum()),
    'ApprovalFY_malformed_text': int(fy_malformed.sum()),
    'ApprovalDate_parse_fail': int(eda['ApprovalDate'].isna().sum()),
    'ApprovalFY_vs_expected_federal_fiscal_year_mismatch': int(year_mismatch.sum()),
    'ApprovalDate_1960s_century_rollover_corrected': int(date_century_corrections['ApprovalDate']),
    'Approved_amount_missing_or_nonpositive': int(eda['GrAppv'].isna().sum() + eda['GrAppv'].le(0).sum()),
    'SBA_approval_above_total_approval': int((eda['SBA_Appv'] > eda['GrAppv']).sum()),
    'Actual_disbursement_missing_or_nonpositive': int(eda['DisbursementGross'].isna().sum() + eda['DisbursementGross'].le(0).sum()),
    'ApprovalYear_before_1987': int(eda['ApprovalYear'].lt(1987).sum()),
    'ApprovalYear_after_2014': int(eda['ApprovalYear'].gt(2014).sum()),
    'Unknown_RevLineCr': int(eda['RevLineCr_clean'].eq('Unknown').sum()),
    'Unknown_LowDoc': int(eda['LowDoc_clean'].eq('Unknown').sum()),
}
quality = pd.DataFrame(list(audit_checks.items()), columns=['quality_check', 'rows_or_count'])
quality.to_csv(audit_dir / '03_quality_checks.csv', index=False)
print('Quality checks (flags, not automatic deletions):')
display(quality)

status_counts = eda['MIS_Status'].fillna('Missing/unknown').value_counts(dropna=False)
status_counts.rename_axis('status').reset_index(name='loans').to_csv(
    audit_dir / '04_status_counts.csv', index=False
)

## Save the two cleaned datasets

The **EDA file** is the fuller version for exploring the historical loans. The **model file** is smaller and leaves out fields that would give away what happened after approval, like charge-off dates and amounts.

For the model, we're only keeping loans where `MIS_Status` is known. That means **1,997 loans won't be used for training**, but they're still in the EDA file. The code checks that the numbers add up before saving anything.

One detail for the model teammate: `GrAppv`, `SBA_Appv`, and the guarantee share assume the approved loan terms are known when we make the prediction. If the question changes to *before* those terms are decided, leave those columns out.

In [ ]:
MODEL_FEATURES = [
    'State','BankState','Term','NoEmp','NewExist_clean','UrbanRural_clean',
    'NAICS_Sector','RevLineCr_clean','LowDoc_clean','InState_Bank',
    'GrAppv','SBA_Appv','SBA_Guarantee_Share'
]
# Keep loan IDs and years for tracking and splitting, not prediction.
MODEL_METADATA = ['LoanNr_ChkDgt', 'ApprovalYear']
MODEL_TARGET = 'Default'

# Only loans with a known status go into the modeling file.
model = eda.loc[eda['Default'].notna(),
                MODEL_METADATA + MODEL_FEATURES + [MODEL_TARGET]].copy()
model[MODEL_TARGET] = model[MODEL_TARGET].astype('int8')

LEAKAGE_BLOCKLIST = {
    'MIS_Status','ChgOffDate','ChgOffPrinGr','BalanceGross',
    'DisbursementDate','DisbursementGross','Name','Bank','CreateJob',
    'RetainedJob','ApprovalDate','City','ZIP5'
}
assert set(MODEL_FEATURES).isdisjoint(LEAKAGE_BLOCKLIST), 'Leaky predictor included.'
assert model[MODEL_TARGET].isin([0, 1]).all()
assert len(eda) == n_original
assert len(model) + int(eda['Default'].isna().sum()) == n_original
if eda['LoanNr_ChkDgt'].isna().any() or eda['LoanNr_ChkDgt'].duplicated().any():
    raise AssertionError('Missing/duplicate loan IDs. Investigate before handoff.')
if eda['MIS_Status'].notna().sum() != len(model):
    raise AssertionError('Status/model row reconciliation failed.')

eda_path = processed_dir / 'sba_eda_ready.csv'
model_path = processed_dir / 'sba_model_candidates.csv'
eda.to_csv(eda_path, index=False, date_format='%Y-%m-%d')
model.to_csv(model_path, index=False)

reconciliation = pd.DataFrame([
    ('Raw rows', n_original),
    ('EDA-ready rows (all original rows)', len(eda)),
    ('Excluded only from supervised model (missing/unknown status)', int(eda.Default.isna().sum())),
    ('Model-candidate rows (known outcome)', len(model)),
    ('Raw columns', raw.shape[1]),
    ('Baseline model feature columns', len(MODEL_FEATURES)),
], columns=['item','count'])
reconciliation.to_csv(audit_dir / '05_reconciliation.csv', index=False)
print('SAVED:', eda_path)
print('SAVED:', model_path)
display(reconciliation)

## Notes for whoever uses the files next

**EDA and charts:** Use `sba_eda_ready.csv`. If you're calculating a default rate, only count loans with `PIF` or `CHGOFF` as their outcome. Include the number of loans behind each percentage.

**Logistic regression:** Use `sba_model_candidates.csv`. The outcome to predict is `Default`. The exact input columns are in `06_model_feature_contract.json`. Loan ID and approval year are included for reference, not as regular model inputs.

Split into training and testing data **before** fitting missing-value replacements, scaling, or category encoding. Otherwise, information from the test set could leak into training.

**Financial analysis:** Use the EDA file for actual disbursement amounts. The case study uses a simplified exercise assumption of **+5%** for loans repaid and **−25%** for defaults. These aren't actual bank profit margins.

The original data only shows loans that were approved. We can't use it to know for sure what would've happened to loans that were rejected.

In [ ]:
contract = {
    'dataset': 'sba_model_candidates.csv',
    'target': MODEL_TARGET,
    'positive_label': 1,
    'positive_meaning': 'CHGOFF (loan charged off/default)',
    'negative_label': 0,
    'negative_meaning': 'PIF (loan paid in full)',
    'features': MODEL_FEATURES,
    'metadata_exclude_as_baseline_predictors': MODEL_METADATA,
    'approved_terms_caveat': 'GrAppv, SBA_Appv, and SBA_Guarantee_Share assume approved terms are known at the scoring point. Remove them for earlier pre-underwriting decisions.',
    'fit_imputation_encoding_scaling_after_train_test_split': True,
    'excluded_post_approval_or_outcome_fields': sorted(LEAKAGE_BLOCKLIST),
    'status_missing_treatment': 'Excluded from supervised target model only, retained in EDA export.',
    'excluded_not_by_default': 'Outliers, unknown NAICS, invalid codes, and missing values are not blindly discarded.',
    'selection_bias': 'Only approved loans are represented, outcomes of applications never approved are not observed.'
}
(audit_dir / '06_model_feature_contract.json').write_text(json.dumps(contract, indent=2))

dictionary = pd.DataFrame([
    ('LoanNr_ChkDgt', 'Stable loan identifier', 'Join only, NEVER train'),
    ('MIS_Status', 'Historical repayment outcome', 'EDA label, NEVER train'),
    ('Default', 'Binary outcome: 1=CHGOFF, 0=PIF', 'Train TARGET only'),
    ('ApprovalYear', 'Approval date year / fiscal-year fallback', 'Split/cohort check, exclude baseline predictor'),
    ('Term', 'Approved loan duration (months)', 'Candidate predictor'),
    ('NoEmp', 'Reported number of employees', 'Candidate predictor'),
    ('State', 'Borrower state', 'Candidate predictor'),
    ('BankState', 'Lender state', 'Candidate predictor'),
    ('InState_Bank', 'Borrower/lender states match, when both known', 'Candidate predictor'),
    ('NAICS_Sector', 'First two NAICS digits or Unknown', 'Candidate predictor'),
    ('NewExist_clean', 'Existing/New/Unknown business category', 'Candidate predictor'),
    ('UrbanRural_clean', 'Urban/Rural/Unknown', 'Candidate predictor'),
    ('LowDoc_clean', 'Y/N/Unknown, nonstandard codes not guessed', 'Candidate predictor'),
    ('RevLineCr_clean', 'Y/N/Unknown, nonstandard codes not guessed', 'Candidate predictor'),
    ('GrAppv', 'Gross approved loan amount', 'Approved-terms candidate, document scoring point'),
    ('SBA_Appv', 'SBA-guaranteed approved amount', 'Approved-terms candidate, document scoring point'),
    ('SBA_Guarantee_Share', 'SBA_Appv / GrAppv when in [0,1]', 'Approved-terms candidate, document scoring point'),
    ('DisbursementGross', 'Actual amount disbursed', 'Financial/EDA only, POST-approval'),
    ('ChgOffPrinGr', 'Charged-off principal', 'EDA only, reveals outcome'),
    ('ChgOffDate', 'Date of charge-off', 'EDA only, reveals outcome'),
    ('BalanceGross', 'Recorded loan balance', 'EDA only, outcome-related'),
    ('DisbursementDate', 'Actual disbursement date', 'EDA only, post-approval'),
    ('CreateJob', 'Reported jobs to create', 'EDA only until timing verified'),
    ('RetainedJob', 'Reported retained jobs', 'EDA only until timing verified'),
], columns=['variable', 'interpretation', 'allowed_use'])
dictionary.to_csv(audit_dir / '07_data_dictionary.csv', index=False)

# Save the row counts and filenames so everyone can check the handoff.
summary = {
    'source_name': raw_path.name,
    'source_rows': int(n_original),
    'eda_rows': int(len(eda)),
    'model_labeled_rows': int(len(model)),
    'target_1_defaults': int(model['Default'].sum()),
    'target_0_paid': int((model['Default'] == 0).sum()),
    'target_missing_excluded_from_model': int(eda['Default'].isna().sum()),
    'outputs': {'eda': str(eda_path.relative_to(repo_root)), 'model_candidates': str(model_path.relative_to(repo_root))},
    'post_approval_leakage_prevented': True
}
(audit_dir / '08_handoff_summary.json').write_text(json.dumps(summary, indent=2))
print('Handoff documentation saved to:', audit_dir)
display(pd.DataFrame([summary]).drop(columns=['outputs']))

## Last checks before sharing

- [ ] Run every cell and make sure there are no errors.
- [ ] Check that `05_reconciliation.csv` accounts for all 899,164 loans.
- [ ] Look through `03_quality_checks.csv` before treating unusual values as real patterns.
- [ ] Share `sba_eda_ready.csv` with the EDA and finance teammates.
- [ ] Share `sba_model_candidates.csv` and `06_model_feature_contract.json` with the modeling teammate.
- [ ] Keep the big CSVs out of normal Git. The notebook and small audit files can go in our repo.

**Short version of what I did:** cleaned up the formatting, kept track of missing and unusual values, and made a separate dataset for the model so it doesn't accidentally use future information.